# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/notebooks/meeting17_neural_networks.ipynb)

# BME i9400 — Meeting 17
## Neural Networks

**Fall 2026 · Wednesday, November 4**

Today's lecture is about the multilayer perceptron, the simplest neural network. It builds the model
from logistic regression, derives the forward and backward passes, trains a network written from
scratch, and examines how the size of the network and its regularisation control what it learns. It
closes by comparing a neural network with logistic regression on the clinical data of Meetings 8 to 12.

**Agenda**

1. Beyond a linear boundary
2. The multilayer perceptron
3. The forward pass
4. The backward pass
5. Training a network
6. Capacity
7. Regularisation
8. A neural network on the clinical data
9. Connection to the course

---
## 1 — Beyond a linear boundary

Logistic regression draws a straight decision boundary (Meeting 8, section 5): the set of points where
$x^{\top} w = 0$. Many clinical relationships are not of this form. Risk often rises when a measurement
is **abnormal in either direction** — too low or too high blood pressure, sodium, or temperature — and
no straight line separates "normal" from "abnormal" when abnormal lies on both sides.

The cell below simulates such a problem:

- 400 patients, two standardised measurements $x_1$ and $x_2$
- the probability of the outcome rises with the distance of $(x_1, x_2)$ from the centre of an
  elliptical "normal range", and the outcome is drawn from that probability
- half of the patients form a training set and half a validation set

> The data are simulated so that the true boundary is known. The clinical dataset returns in
> section 8.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

rng = np.random.default_rng(17)
X   = rng.normal(size=(400, 2))
r   = np.sqrt((X[:, 0] / 1.2) ** 2 + (X[:, 1] / 0.9) ** 2)        # distance from the centre of the normal range
y   = (rng.random(400) < 1 / (1 + np.exp(-4 * (r - 1.3)))).astype(int)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)

linear  = LogisticRegression().fit(X_tr, y_tr)
squares = LogisticRegression().fit(np.column_stack([X_tr, X_tr ** 2]), y_tr)     # hand-made features x1^2, x2^2
print(f"prevalence {y.mean():.3f};  predicting the majority class gives accuracy {1 - y_va.mean():.3f}\n")
print(f"logistic regression on x1, x2:                validation accuracy {linear.score(X_va, y_va):.3f}")
print(f"logistic regression on x1, x2, x1^2, x2^2:    validation accuracy "
      f"{squares.score(np.column_stack([X_va, X_va ** 2]), y_va):.3f}")

In [ ]:
def boundary(ax, predict, title, X=X_tr, y=y_tr, lim=3.6):
    g = np.linspace(-lim, lim, 250)
    A, B = np.meshgrid(g, g)
    P = predict(np.column_stack([A.ravel(), B.ravel()])).reshape(A.shape)
    ax.contourf(A, B, P, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.35)
    ax.contour(A, B, P, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=10, color="#2b6cb0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=10, color="#c1272d")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(title, fontsize=11); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
boundary(axes[0], lambda Z: linear.predict_proba(Z)[:, 1], "logistic regression on $x_1, x_2$")
boundary(axes[1], lambda Z: squares.predict_proba(np.column_stack([Z, Z ** 2]))[:, 1],
         "logistic regression on $x_1, x_2, x_1^2, x_2^2$")
plt.tight_layout(); plt.show()

On the two measurements alone, logistic regression does no better than predicting the majority
class: no straight line separates the ring of positive patients (red) from the negatives at the
centre (blue). Adding the squares $x_1^2$ and $x_2^2$ as features lets the same linear model draw an
ellipse, and the validation accuracy rises to $0.85$.

The second model worked because the right features were chosen by hand, from knowledge of how the data
were generated. With real data that knowledge is usually missing. A **neural network** learns the
features from the data, as part of the fit.

---
## 2 — The multilayer perceptron

Logistic regression computes one weighted sum of the inputs and passes it through the sigmoid. A
**multilayer perceptron** (MLP) with one hidden layer first computes $m$ weighted sums, passes each
through a nonlinear function, and then applies logistic regression to the results:

$$a = W_1 x + b_1, \qquad h = \phi(a), \qquad z = w_2^{\top} h + b_2, \qquad p = \sigma(z)$$

- $x$: the $d$ input features
- $W_1$: an $m \times d$ matrix of weights, and $b_1$: $m$ biases — row $j$ of $W_1$ defines
  **hidden unit** $j$
- $a$: the $m$ pre-activations, and $h = \phi(a)$: the $m$ **hidden activations**, the learned features
- $w_2$, $b_2$: the weights and bias of the output unit, a logistic regression on $h$
- $p$: the predicted probability of the positive class

> A **unit** (or "neuron") is one weighted sum followed by a nonlinearity. A **layer** is a set of
> units that all receive the same inputs. "Hidden" means that the layer's values are neither inputs
> nor outputs: nothing in the data says what they should be.

### The activation function

$\phi$ is applied to each pre-activation separately. The standard choice is the **rectified linear
unit**:

$$\phi(a) = \text{ReLU}(a) = \max(a, 0)$$

> ReLU passes positive values unchanged and sets negative values to zero. Its derivative is $1$ for
> $a > 0$ and $0$ for $a < 0$. Older networks used the sigmoid or $\tanh$, whose derivatives become
> very small for large $|a|$, which slows training.

**Why the nonlinearity is necessary.** Without $\phi$, the hidden layer would be linear, and

$$z = w_2^{\top}(W_1 x + b_1) + b_2 = (w_2^{\top} W_1)\, x + (w_2^{\top} b_1 + b_2) = \tilde w^{\top} x + \tilde b$$

which is logistic regression again, with weights $\tilde w = W_1^{\top} w_2$ and bias
$\tilde b = w_2^{\top} b_1 + b_2$. A stack of linear layers is one linear layer. The nonlinearity is
what lets the network represent boundaries that are not straight.

**Parameters.** $W_1$ has $md$ entries, $b_1$ has $m$, $w_2$ has $m$, and $b_2$ is one number:
$m(d + 2) + 1$ in total. With $d = 2$ inputs and $m = 8$ hidden units, that is $33$.

---
## 3 — The forward pass

Computing $p$ from $x$ is the **forward pass**. For $n$ samples at once, the inputs are the rows of an
$n \times d$ matrix $X$, and every step becomes a matrix operation:

| quantity | computation | shape |
|---|---|---|
| pre-activations | $A = X W_1^{\top} + b_1$ | $n \times m$ |
| hidden activations | $H = \max(A, 0)$ | $n \times m$ |
| output scores | $z = H w_2 + b_2$ | $n$ |
| probabilities | $p = \sigma(z)$ | $n$ |

> Adding the vector $b_1$ to the matrix $X W_1^{\top}$ adds it to every row; NumPy does this
> automatically (**broadcasting**).

The weights must start somewhere. They are drawn at random, with a spread that keeps the activations
on a sensible scale as they pass through the layer; section 5 shows why they cannot all start at zero.

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def init(m, d=2, seed=0):
    # Random starting weights: standard deviation sqrt(2 / inputs) for a ReLU layer, sqrt(1 / inputs) for the output.
    g = np.random.default_rng(seed)
    return {"W1": g.normal(0, np.sqrt(2 / d), (m, d)), "b1": np.zeros(m),
            "w2": g.normal(0, np.sqrt(1 / m), m),      "b2": 0.0}

def forward(P, X):
    A = X @ P["W1"].T + P["b1"]           # pre-activations,     n x m
    H = np.maximum(A, 0)                  # hidden activations,  n x m
    p = sigmoid(H @ P["w2"] + P["b2"])    # probabilities,       n
    return A, H, p

def cross_entropy(P, X, y):
    p = np.clip(forward(P, X)[2], 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

P = init(m=8)
A, H, p = forward(P, X_tr)
print(f"shapes:  X {X_tr.shape}   W1 {P['W1'].shape}   A {A.shape}   H {H.shape}   p {p.shape}")
print(f"parameters: {sum(np.size(v) for v in P.values())}")
print(f"before training: cross-entropy {cross_entropy(P, X_tr, y_tr):.3f}   (log 2 = {np.log(2):.3f} is a coin flip)")

---
## 4 — The backward pass

Training by gradient descent (Meeting 13) needs the gradient of the loss with respect to every weight.
The **backward pass** computes it with the chain rule, working from the output back toward the inputs.

For one sample, the loss is the cross-entropy of Meeting 8,
$\ell = -\left[\, y \log p + (1 - y)\log(1 - p) \,\right]$, and Meeting 8 showed that

$$\frac{\partial \ell}{\partial z} = p - y \;\equiv\; \delta_2$$

> $\delta_2$ is the **error signal** at the output: positive when the model's probability is too high,
> negative when it is too low.

**The output layer.** $z = w_2^{\top} h + b_2$, so $\partial z / \partial w_2 = h$ and
$\partial z / \partial b_2 = 1$:

$$\frac{\partial \ell}{\partial w_2} = \frac{\partial \ell}{\partial z}\,\frac{\partial z}{\partial w_2} = \delta_2\, h, \qquad \frac{\partial \ell}{\partial b_2} = \delta_2$$

This is the logistic-regression gradient of Meeting 8, with the hidden activations in place of the
inputs.

**Back to the hidden layer.** $z$ depends on $h_j$ through the weight $w_{2,j}$, so
$\partial \ell / \partial h = \delta_2\, w_2$. Each $h_j = \phi(a_j)$, so

$$\frac{\partial \ell}{\partial a_j} = \frac{\partial \ell}{\partial h_j}\,\frac{\partial h_j}{\partial a_j} = \delta_2\, w_{2,j}\, \phi'(a_j) \;\equiv\; \delta_{1,j}$$

> $\phi'(a_j)$ is $1$ if $a_j > 0$ and $0$ otherwise. A hidden unit that was inactive for this sample
> passes no error back, and its incoming weights are not changed by it.

**The hidden layer's weights.** $a = W_1 x + b_1$, so $\partial a_j / \partial W_{1,jk} = x_k$:

$$\frac{\partial \ell}{\partial W_1} = \delta_1\, x^{\top} \quad (m \times d), \qquad \frac{\partial \ell}{\partial b_1} = \delta_1$$

Averaged over the $n$ samples, with $\Delta_1$ the $n \times m$ matrix whose rows are the $\delta_1$:

$$\nabla_{w_2} L = \tfrac{1}{n} H^{\top} \delta_2, \qquad \nabla_{W_1} L = \tfrac{1}{n} \Delta_1^{\top} X$$

> **Backpropagation** (Rumelhart, Hinton, and Williams, *Nature*, 1986) is this procedure: one
> forward pass stores $A$ and $H$, and one backward pass reuses them to give every gradient, at about
> the cost of a second forward pass. Deeper networks repeat the hidden-layer step once per layer.

In [ ]:
def backward(P, X, y):
    # Gradients of the mean cross-entropy with respect to every parameter.
    A, H, p = forward(P, X)
    n  = len(y)
    d2 = (p - y) / n                              # output error signal, averaged
    D1 = np.outer(d2, P["w2"]) * (A > 0)          # hidden error signals: delta_2 * w2 * phi'(a),  n x m
    return {"W1": D1.T @ X, "b1": D1.sum(axis=0), "w2": H.T @ d2, "b2": d2.sum()}

# Check two entries against a finite-difference approximation: (L(w + eps) - L(w - eps)) / (2 eps).
grads, eps = backward(P, X_tr, y_tr), 1e-6
for name, idx in [("W1", (0, 1)), ("b1", (2,)), ("w2", (3,))]:
    up, dn = {k: np.array(v, dtype=float) for k, v in P.items()}, {k: np.array(v, dtype=float) for k, v in P.items()}
    up[name][idx] += eps; dn[name][idx] -= eps
    numeric = (cross_entropy(up, X_tr, y_tr) - cross_entropy(dn, X_tr, y_tr)) / (2 * eps)
    print(f"d loss / d {name}{list(idx)}:   backpropagation {grads[name][idx]:+.7f}   finite difference {numeric:+.7f}")

The gradients from backpropagation agree with the finite-difference approximations to at least six
decimal places.

> A **gradient check** of this kind is the standard test of a hand-written backward pass: perturb one
> weight up and down by a small amount, and compare the change in the loss with the gradient. It costs
> two forward passes per weight, so it is used only for testing. Libraries such as PyTorch (Meeting 18)
> compute the backward pass automatically from the forward pass, which is called **automatic
> differentiation**.

---
## 5 — Training a network

Training is the loop of Meeting 13: forward pass, backward pass, step against the gradient. The cell
below trains the 8-unit network for 3,000 steps of full-batch gradient descent with $\eta = 0.5$, from
three starting points:

- random weights, seed 0
- random weights, seed 3
- every weight equal to zero

In [ ]:
def train(P, steps=3000, eta=0.5):
    history = []
    for _ in range(steps):
        g = backward(P, X_tr, y_tr)
        P = {k: P[k] - eta * g[k] for k in P}
        history.append(cross_entropy(P, X_tr, y_tr))
    return P, np.array(history)

accuracy = lambda P, X, y: np.mean((forward(P, X)[2] >= 0.5) == y)
zeros    = {"W1": np.zeros((8, 2)), "b1": np.zeros(8), "w2": np.zeros(8), "b2": 0.0}
nets = {"random start, seed 0": train(init(8, seed=0)),
        "random start, seed 3": train(init(8, seed=3)),
        "all weights zero":     train(zeros)}
for name, (Pn, h) in nets.items():
    distinct = len(np.unique(np.round(Pn["W1"], 6), axis=0))
    print(f"{name:<22} final loss {h[-1]:.4f}   training acc {accuracy(Pn, X_tr, y_tr):.3f}   "
          f"validation acc {accuracy(Pn, X_va, y_va):.3f}   distinct hidden units {distinct}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.9))
for ax, (name, (Pn, h)) in zip(axes[:2], list(nets.items())[:2]):
    boundary(ax, lambda Z, Pn=Pn: forward(Pn, Z)[2], f"8 hidden units, {name}\nvalidation accuracy {accuracy(Pn, X_va, y_va):.3f}")
for (name, (Pn, h)), col in zip(nets.items(), ["#5b3a8e", "#2f8f5b", "#c1272d"]):
    axes[2].plot(h, lw=2, color=col, label=name)
axes[2].set_xscale("log"); axes[2].set_xlabel("gradient descent step"); axes[2].set_ylabel("training cross-entropy")
axes[2].set_title("Training loss"); axes[2].legend(fontsize=10); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

From a random start the network learns an elliptical boundary close to the true one, with a validation
accuracy of about $0.86$, as good as the hand-made squared features of section 1 — without being told
about squares. The boundary is a polygon: each ReLU unit is a linear function on one side of a line
and zero on the other, so the network's score is piecewise linear, and its boundary is made of
straight segments, one or more per hidden unit.
Two other results follow from the structure of the model.

**The answer depends on the start.** Seeds 0 and 3 reach different final losses and somewhat different
boundaries. The loss of a network with a hidden layer is not convex (Meeting 13, section 6), and
gradient descent finds a different local minimum from each starting point.

**Starting at zero fails.** With every weight zero, every hidden unit computes the same thing and
receives the same gradient, so they remain identical forever — and with ReLU at zero, the gradient
reaching them is zero, so they never move at all. The network stays at the majority-class answer.

> Random initialisation **breaks the symmetry** between hidden units, so that each can learn a
> different feature. The scale of the random weights matters too: too large and the activations grow
> from layer to layer, too small and they shrink to zero. The $\sqrt{2/\text{inputs}}$ rule of section
> 3 (He initialisation) keeps them on a constant scale through ReLU layers.

---
## 6 — Capacity

The number of hidden units sets how complex a boundary the network can draw — its **capacity**. The
cell below fits networks with 1, 3, 10, and 100 hidden units with scikit-learn's `MLPClassifier`.

> `MLPClassifier(hidden_layer_sizes=(m,))` is the network of section 2 with $m$ ReLU hidden units.
> `solver="lbfgs"` is a second-order optimiser suited to small datasets; `alpha` is an L2 penalty on
> the weights, set almost to zero here so that capacity alone is varied.
>
> A network with one hidden layer and enough units can approximate any continuous function on a bounded
> region as closely as required (Hornik, Stinchcombe, and White, *Neural Networks*, 1989). This
> **universal approximation** result says the right network exists; it says nothing about whether
> training on finite data will find it, or how many units it needs.

In [ ]:
from sklearn.neural_network import MLPClassifier

capacity = {}
for m in [1, 3, 10, 100]:
    net = MLPClassifier(hidden_layer_sizes=(m,), solver="lbfgs", alpha=1e-4, max_iter=5000, random_state=0)
    capacity[m] = net.fit(X_tr, y_tr)
    print(f"{m:>3} hidden units  ({m * 4 + 1:>3} parameters):   training accuracy {net.score(X_tr, y_tr):.3f}   "
          f"validation accuracy {net.score(X_va, y_va):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, (m, net) in zip(axes, capacity.items()):
    boundary(ax, lambda Z, net=net: net.predict_proba(Z)[:, 1],
             f"{m} hidden unit{'s' if m > 1 else ''}\ntrain {net.score(X_tr, y_tr):.2f}   validation {net.score(X_va, y_va):.2f}")
plt.tight_layout(); plt.show()

One hidden unit can bend the boundary only once, and it underfits. Three units are enough to enclose
the centre, and the validation accuracy reaches its best range. With 100 units and almost no penalty,
the network classifies every training patient correctly by drawing islands around individual points,
and the validation accuracy falls: it has fitted the noise in 200 patients. This is the bias–variance
trade-off of Meeting 11, with the number of hidden units in the role of the number of features.

---
## 7 — Regularisation

Networks are usually made larger than necessary and then regularised, rather than sized exactly. Three
methods are standard:

- **Weight decay**: an L2 penalty $\alpha \|w\|^2$ on all weights, as in ridge regression (Meeting 11).
  It is the `alpha` of `MLPClassifier`, and the `weight_decay` of PyTorch's optimisers.
- **Early stopping**: stop training when the validation loss stops improving (Meeting 13, section 7).
- **Dropout**: during training, set each hidden activation to zero at random with probability $q$ at
  every step, so that no unit can rely on any particular other unit; at prediction time use all units.

> Each method has a hyperparameter — $\alpha$, the number of epochs, $q$ — chosen on validation data or
> by cross-validation, with the optimism of Meeting 12 if the same data are used to report the result.

The cell below fits the 100-unit network to only the first 60 training patients, at three values of
$\alpha$, and evaluates on the same 200 validation patients.

In [ ]:
decay = {}
for alpha in [1e-4, 1.0, 10.0]:
    net = MLPClassifier(hidden_layer_sizes=(100,), solver="lbfgs", alpha=alpha, max_iter=5000, random_state=0)
    decay[alpha] = net.fit(X_tr[:60], y_tr[:60])
    print(f"alpha = {alpha:<7g}  training accuracy {net.score(X_tr[:60], y_tr[:60]):.3f}   "
          f"validation accuracy {net.score(X_va, y_va):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.7))
for ax, (alpha, net) in zip(axes, decay.items()):
    boundary(ax, lambda Z, net=net: net.predict_proba(Z)[:, 1],
             f"100 units, 60 patients, $\\alpha$ = {alpha:g}\nvalidation accuracy {net.score(X_va, y_va):.2f}",
             X=X_tr[:60], y=y_tr[:60])
plt.tight_layout(); plt.show()

With almost no penalty, 100 hidden units fit the 60 training patients perfectly and generalise
poorly. A moderate penalty, $\alpha = 1$, keeps the weights small, the boundary becomes a smooth closed
curve, and the validation accuracy rises substantially even though the training accuracy falls. A
penalty that is too strong, $\alpha = 10$, flattens the network almost to a constant and it underfits.
The pattern is that of ridge regression in Meeting 11, section 2.

---
## 8 — A neural network on the clinical data

The question for the course: does a neural network improve on logistic regression for the diabetes
classifier of Meetings 8 to 12? The cell below runs the Meeting 11 pipeline — median imputation,
standardisation, model — with 5-fold cross-validation on the same folds, for logistic regression and
four networks: one or two hidden layers of 100 units, with weak and with moderate weight decay.

> `hidden_layer_sizes=(100, 100)` stacks two hidden layers: the second takes the first's activations
> as its inputs. The default solver, `adam` (Meeting 13, section 6), is used here, as it would be for a
> larger dataset.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

pima = pd.read_csv("https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv")
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
pima[cols] = pima[cols].replace(0, np.nan)
Xp, yp = pima.drop(columns="Outcome"), pima["Outcome"].values
folds  = StratifiedKFold(5, shuffle=True, random_state=9400)                     # Meeting 11's folds
print(f"{len(yp)} women, {yp.sum()} with diabetes, {Xp.shape[1]} measurements")

In [ ]:
candidates = {"logistic regression":            LogisticRegression(max_iter=1000)}
for layers in [(100,), (100, 100)]:
    for alpha in [1e-4, 1.0]:
        candidates[f"MLP {layers}, alpha = {alpha:g}"] = MLPClassifier(layers, alpha=alpha, max_iter=2000, random_state=0)

for name, clf in candidates.items():
    s = cross_val_score(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clf),
                        Xp, yp, cv=folds, scoring="roc_auc")
    print(f"{name:<32} 5-fold CV AUC {s.mean():.3f}   (SE {s.std(ddof=1) / np.sqrt(5):.3f})")

The networks do not improve on logistic regression. With moderate weight decay they match it to within
about one standard error, and the best of them is the best of four configurations, which is
optimistic by the argument of Meeting 12. Without the penalty, the two-layer network — about 11,000
weights fitted to about 600 women per fold — is clearly worse.

This is the usual result for clinical tabular data. The relationship between eight routine
measurements and diabetes is close to linear in the log-odds, there are a few hundred patients, and a
model with more capacity has more variance to pay for and little bias to remove. A systematic review
of 71 clinical prediction studies found no performance benefit of machine learning methods over
logistic regression when the comparison was done fairly (Christodoulou et al., *Journal of Clinical
Epidemiology*, 2019).

> Neural networks earn their cost where the inputs are high-dimensional and the useful features are
> unknown — images, signals, text (Meetings 21 to 24) — and where there are many thousands of examples.
> For a capstone on tabular data, logistic regression is the baseline that a network must beat, under
> nested cross-validation, before it is preferred.

---
## 9 — Connection to the course

**Everything so far, assembled.** The output unit is the logistic regression of Meeting 8; the loss is
its cross-entropy; the training is the gradient descent of Meeting 13, with its learning rate, its
non-convexity, and its early stopping; capacity and weight decay are the bias–variance trade-off and
the ridge penalty of Meeting 11; and the comparison in section 8 is evaluated with the folds of
Meeting 11 and the caution of Meeting 12.

**What is new.** The hidden layer learns features, and backpropagation makes it possible to compute the
gradient for every weight at the cost of one extra pass.

**Next.** Meeting 18 rebuilds today's network in PyTorch, where the backward pass is automatic and the
training loop — batches, epochs, learning rate, early stopping — is written explicitly, and diagnoses
the training runs that go wrong. Meetings 21 to 24 replace the fully connected hidden layer with
layers designed for images, signals, and text.

## What to take away

- An MLP computes learned features $h = \phi(W_1 x + b_1)$ and applies logistic regression to them
- Without a nonlinearity, any number of layers collapses to one linear model; ReLU is the standard choice
- The forward pass computes and stores the activations; the backward pass applies the chain rule from
  the output back, reusing them: $\delta_2 = p - y$, $\delta_1 = \delta_2\, w_2 \odot \phi'(a)$
- Check a hand-written backward pass against finite differences
- Initialise at random, never at zero; the loss is non-convex, and the result depends on the start
- The number of hidden units sets the capacity; regularise with weight decay, early stopping, or dropout
- On small clinical tabular data a network rarely beats logistic regression; compare them under nested
  cross-validation

### Monday, November 9 — Studio

A PyTorch MLP on clinical tabular data: the network of today, written with automatic differentiation
and an explicit training loop, together with the loss curves that reveal underfitting, overfitting,
and unstable training. The capstone briefs are released at this meeting.

## Recommended resources

Everything here is optional and freely available unless marked **(CCNY)**. Each item covers material
from this lecture in a different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[But what is a neural network?](https://www.youtube.com/watch?v=aircAruvnKk)** — 3Blue1Brown | Layers, weights, and activations, drawn. | 18 min |
| **[Backpropagation, intuitively](https://www.youtube.com/watch?v=Ilg3gGewQ5U)** — 3Blue1Brown | Section 4 without the notation. | 13 min |
| **[The Essential Main Ideas of Neural Networks](https://www.youtube.com/watch?v=CqOfi41LfDw)** — StatQuest | A network that bends a curve, worked by hand. | 19 min |
| **[TensorFlow Playground](https://playground.tensorflow.org/)** | Train a small network in the browser; vary the hidden units, activation, and regularisation of sections 6 and 7. | 20 min |

### Going further

| Resource | Why this one | Time |
|---|---|---|
| **[Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/)** — Nielsen, chapters 1 and 2 | Backpropagation derived carefully, with code. Free online. | 3 h |
| **[Deep Learning, chapter 6: Deep Feedforward Networks](https://www.deeplearningbook.org/contents/mlp.html)** — Goodfellow, Bengio, Courville | The MLP, activations, and back-propagation, at a graduate level. | 2 h |
| **[Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html)** — Srivastava et al., *JMLR*, 2014 | The original dropout paper. Open access. | 1 h |
| **[A systematic review shows no performance benefit of machine learning over logistic regression for clinical prediction models](https://doi.org/10.1016/j.jclinepi.2019.02.004)** — Christodoulou et al., *J Clin Epidemiol*, 2019 **(CCNY)** | The evidence behind section 8. | 45 min |
| **[Why do tree-based models still outperform deep learning on tabular data?](https://arxiv.org/abs/2207.08815)** — Grinsztajn, Oyallon, Varoquaux, 2022 | A benchmark of networks against tree ensembles (Meeting 19) on tabular data. | 1 h |
| **[Learning representations by back-propagating errors](https://doi.org/10.1038/323533a0)** — Rumelhart, Hinton, Williams, *Nature*, 1986 **(CCNY)** | The paper that introduced backpropagation to neural networks; four pages. | 30 min |